# Day 19 — Constraints, Indexes and Upserts: NOT NULL, UNIQUE, CHECK, ON CONFLICT, Indexes, EXPLAIN ANALYZE & Migrations (12 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: make the database protect itself and make your loader safe to re-run. Constraints stop bad data at the door, upserts turn "insert" into "insert or update", indexes make the slow report fast, and EXPLAIN ANALYZE proves it with numbers. Migrations tie it together: every schema change is a numbered file that runs once, in order, everywhere.

- Deliverable: day19/upsert_loader.py, a nightly loader for a CMS JSON export that applies pending migrations, upserts posts, authors and tags in one transaction, and can run any number of times with the same result. Plus two migrations you write (an index and a new rule), explain_notes.md with the EXPLAIN ANALYZE output before and after your index, and day19/dsa.py.

- Where this sits: Week 3, Phase 1 (Foundation), day 5 of 7. Day 18's loader failed with UniqueViolation the second time it ran; today fixes that properly. Tomorrow (Day 20) adds analytics SQL on top of these tables, Week 4's FastAPI app relies on these constraints for its error responses, and Week 6's RAG app re-indexes documents with exactly this upsert pattern.

Time split (5h): the Week 3 split, with the catch-up hour kept.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 5, running every example against setup.sql |
| Hands-on | 2h | upsert_loader.py until the tests pass, migrations 003 and 004, explain_notes.md |
| Catch-up | 1h | Day 18: load_export.py (19 tests) |
| DSA | 0.5h | upsert_records and lower_bound in dsa.py |

Open items first: GitHub main still ends at Day 4's starters, and Days 5 to 18 reached you as patches and zips because the scheduled push is blocked. If you have not applied them yet, do that before anything else (git am on each patch in day order, then git push). Today's folder ships its own data and does not depend on them.

Before you start: Postgres running and the fde database from Day 15. Then, inside day19/:

```bash
# One-time setup for Day 19, run inside day19/ (macOS / Linux shell).
python3 -m venv .venv          # a fresh virtual environment (Day 6)
source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install -r requirements.txt
export DATABASE_URL=postgresql://localhost/fde   # same database as Day 15
psql "$DATABASE_URL" -f setup.sql   # lesson tables: about 2 seconds
```

setup.sql creates two lesson tables: lesson_entries (3 rows, with rules) and lesson_views (300,000 rows of daily view counts, generated so every machine gets the same data). Every output below came from running the code on PostgreSQL 16 with psycopg 3.3.8. Timings will differ on your laptop; the size of the gap is what matters. Re-run setup.sql whenever the lesson data looks wrong.

## Recap (2 minutes, from memory)

Three questions on Day 18. Answer out loud first; the answers are on the last page.

- Why is conn.execute("SELECT * FROM posts WHERE uid = %s", (uid,)) safe while an f-string with the same text is not? What would break if you wrote '%s' with quotes?

- Your loader opens a plain connection (no autocommit), inserts 8 rows and the script ends without conn.commit(). What is in the table afterwards, and why?

- Day 18's loader runs fine on Monday. On Tuesday the scheduler runs it again on the same file. What happens, and which line of the Day 18 hints made sure Tuesday did no damage?

## Concept 1: Constraints, rules the database enforces

### Why it matters for an FDE

Every customer system has more than one writer: your loader, their admin UI, a script someone wrote in 2019, a teammate fixing data by hand in psql. Validation in your Python code only protects the data that goes through your Python code. A constraint is a rule stored in the database itself, so it holds for every writer, forever.

This changes how you debug too. When a customer says "the report shows a negative word count", a CHECK constraint means that cannot happen, and you can rule out a whole class of causes in one sentence. In interviews, "where do you validate data?" has a strong answer: at the edge for good error messages, and in the database as the final guarantee.

### How it works

You met PRIMARY KEY and FOREIGN KEY on Day 17. Today adds the rest of the family. NOT NULL says a column must have a value. UNIQUE says no two rows may share a value (a slug, an email, a CMS uid). CHECK takes any true/false expression about the row: an allowed list of statuses, a number range, even a rule across two columns. DEFAULT is not a rule but fills in a value you leave out. In Contentstack terms these are the field settings "Mandatory", "Unique" and a validation regex, except that they apply to every path into the data, not only the CMS editor.

A row that breaks a rule is refused and the statement fails as a whole. Postgres names the rule in the error, which is why named constraints (CONSTRAINT lesson_entries_status_check CHECK (...)) matter: the name is what you show the user and what you search for in logs. Postgres generates a name if you do not give one, but you want to choose it.

```sql
-- Concept 1: each kind of rule, broken once. Run setup.sql first, then
-- paste these into psql one at a time.

-- NOT NULL: a title is required.
INSERT INTO lesson_entries (uid, title) VALUES ('blt104', NULL);
-- -> ERROR:  null value in column "title" of relation "lesson_entries"
-- ->         violates not-null constraint

-- UNIQUE: no two entries may share a slug (the URL part).
INSERT INTO lesson_entries (uid, title, slug)
VALUES ('blt105', 'Gift cards', 'returns');
-- -> ERROR:  duplicate key value violates unique constraint
-- ->         "lesson_entries_slug_key"
-- -> DETAIL:  Key (slug)=(returns) already exists.

-- CHECK: status must be one of three words. Like an enum field in a CMS.
INSERT INTO lesson_entries (uid, title, status)
VALUES ('blt106', 'FAQ', 'live');
-- -> ERROR:  new row for relation "lesson_entries" violates check
-- ->         constraint "lesson_entries_status_check"

-- CHECK on a number: no negative word counts.
INSERT INTO lesson_entries (uid, title, word_count)
VALUES ('blt107', 'FAQ', -5);
-- -> ERROR:  ... violates check constraint
-- ->         "lesson_entries_word_count_check"

-- DEFAULT fills what you leave out; the row passes every rule.
INSERT INTO lesson_entries (uid, title) VALUES ('blt108', 'FAQ');
SELECT uid, status, word_count, slug FROM lesson_entries
WHERE uid = 'blt108';
-- ->   uid   | status | word_count | slug
-- ->  blt108 | draft  |            |
-- GOTCHA: word_count is NULL and the CHECK still passed. A CHECK only
-- rejects FALSE; NULL >= 0 is unknown, so it is let through. Add
-- NOT NULL as well when "missing" must also be refused.
```

From Python, each kind of violation arrives as its own exception class from psycopg.errors, all children of IntegrityError. The exception's diag attribute holds the structured details Postgres sent, including the constraint name. That is how tonight's loader will tell the content team which rule an entry broke.

In [ ]:
# Concept 1: catch each kind of violation and read WHICH rule failed.
import os

import psycopg
from psycopg import errors  # one exception class per Postgres error code

DB_URL = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")

BAD_ROWS = [
    ("blt201", None, "draft"),       # title missing
    ("blt101", "Copy", "draft"),     # uid already used
    ("blt202", "FAQ", "Published"),  # wrong case: the CHECK is exact
]
INSERT = "INSERT INTO lesson_entries (uid, title, status) VALUES (%s, %s, %s)"

with psycopg.connect(DB_URL, autocommit=True) as conn:
    for row in BAD_ROWS:
        try:
            conn.execute(INSERT, row)
        # All three classes inherit from errors.IntegrityError, so
        # "except errors.IntegrityError" would catch them all at once.
        except (errors.NotNullViolation, errors.UniqueViolation,
                errors.CheckViolation) as e:
            # diag = the structured details Postgres sent with the error
            print(type(e).__name__, "|", e.diag.constraint_name,
                  "|", e.diag.column_name)
# -> NotNullViolation | None | title
# -> UniqueViolation | lesson_entries_pkey | None
# -> CheckViolation | lesson_entries_status_check | None

Real systems need to add rules to tables that already hold years of data. Postgres checks every existing row when you add a constraint, and refuses if any row breaks it. The safe order is always: find the bad rows, agree the fix with the customer, fix them, then add the rule.

```sql
-- Concept 1: adding a rule to a table that already holds data.
-- The customer now wants a word count on every entry.
ALTER TABLE lesson_entries ALTER COLUMN word_count SET NOT NULL;
-- -> ERROR:  column "word_count" of relation "lesson_entries"
-- ->         contains null values

-- Step 1: find the rows that break the rule BEFORE you add it.
SELECT uid, title FROM lesson_entries WHERE word_count IS NULL;
-- ->   uid   |       title
-- ->  blt103 | Draft: size chart

-- Step 2: agree the fix with the customer (here: unknown becomes 0).
UPDATE lesson_entries SET word_count = 0 WHERE word_count IS NULL;
-- -> UPDATE 1

-- Step 3: now the rule goes on, and protects every future row.
ALTER TABLE lesson_entries ALTER COLUMN word_count SET NOT NULL;
-- -> ALTER TABLE

-- A CHECK can use functions: slugs must be lowercase.
ALTER TABLE lesson_entries ADD CONSTRAINT lesson_entries_slug_lower
    CHECK (slug = lower(slug));
-- -> ALTER TABLE

-- \d shows the columns, the rules, and the indexes behind them:
\d lesson_entries
-- -> Indexes:
-- ->     "lesson_entries_pkey" PRIMARY KEY, btree (uid)
-- ->     "lesson_entries_slug_key" UNIQUE CONSTRAINT, btree (slug)
-- -> Check constraints:
-- ->     "lesson_entries_slug_lower" CHECK (slug = lower(slug))
-- ->     "lesson_entries_status_check" CHECK (status = ANY (...))
-- ->     "lesson_entries_word_count_check" CHECK (word_count >= 0)
```

Notice the Indexes section at the end. Every PRIMARY KEY and UNIQUE constraint is backed by an index; that is how Postgres checks uniqueness quickly. Concept 3 builds on this.

### Real-world FDE example

A pharma customer's product pages must never be published without a regulatory approval date. Their CMS UI enforced it, but a bulk-import script did not, and an audit found 14 published pages with no date. The fix was one rule, CHECK (status <> 'published' OR approved_at IS NOT NULL), added after cleaning those 14 rows. From that day no tool, script or person could repeat the mistake. Today's cms_posts table has the same rule, named cms_posts_published_has_date.

### Common mistakes

- Thinking CHECK rejects NULL. It rejects only FALSE; add NOT NULL when a missing value must be refused too.

- Leaving constraints unnamed, so errors say lesson_entries_check1 and nobody knows which rule that is.

- Validating only in application code because "the API is the only writer". It never stays the only writer.

- Adding a constraint to a live table without checking the existing rows first, then watching the migration fail in production.

- Case-sensitive surprises: CHECK (status IN ('draft', ...)) refuses 'Draft'. Normalise in your loader, or the database will say no.

## Concept 2: Upserts with INSERT ... ON CONFLICT

### Why it matters for an FDE

Almost every integration you build is a sync: tonight's CMS export, a CRM pull every 15 minutes, re-embedding documents for a RAG index when they change. Each run sees mostly the same records as last time. A loader that can only INSERT fails on the second run, and one that deletes everything and reloads loses ids, history and anything pointing at those rows.

The property you want is idempotency: running the job twice gives the same result as running it once. Schedulers retry failed jobs, people press the button twice, and a deploy restarts a job halfway. An idempotent job makes all of those harmless. Interviewers ask "what happens if your job runs twice?" on purpose.

### How it works

An upsert is "insert, or update if it already exists". In Postgres you write a normal INSERT and add an ON CONFLICT clause. The conflict target, such as (uid), names a column with a UNIQUE constraint or primary key; that is how Postgres knows which existing row "the same record" means. DO NOTHING skips the row. DO UPDATE SET changes the existing row, and inside it the special name EXCLUDED means the row you tried to insert. If you know JavaScript, EXCLUDED is the incoming object in {...existing, ...incoming}, except you choose field by field which side wins.

```sql
-- Concept 2: three ways to insert a row that may already exist.
-- Re-run setup.sql first.

-- 1. Plain INSERT: the second run of any loader dies here (Day 18).
INSERT INTO lesson_entries (uid, title) VALUES ('blt102', 'Returns');
-- -> ERROR:  duplicate key value violates unique constraint
-- ->         "lesson_entries_pkey"

-- 2. DO NOTHING: skip the row if the uid is taken.
INSERT INTO lesson_entries (uid, title) VALUES ('blt102', 'Returns')
ON CONFLICT (uid) DO NOTHING;
-- -> INSERT 0 0      (0 rows inserted, and no error)

-- 3. DO UPDATE: insert, or change the row that is already there.
--    EXCLUDED = the row you TRIED to insert. "e" = the row in the table.
INSERT INTO lesson_entries AS e (uid, title, word_count)
VALUES ('blt102', 'Returns and refunds', 520)
ON CONFLICT (uid) DO UPDATE
    SET title      = EXCLUDED.title,
        word_count = EXCLUDED.word_count,
        updated_at = now()
RETURNING e.uid, e.title, e.word_count, e.status;
-- ->   uid   |        title        | word_count |  status
-- ->  blt102 | Returns and refunds |        520 | published
-- status was not in SET, so it kept its old value: like a PATCH, not a PUT.

-- The conflict target must be a UNIQUE constraint or the primary key:
INSERT INTO lesson_entries (uid, title) VALUES ('blt109', 'FAQ')
ON CONFLICT (title) DO NOTHING;
-- -> ERROR:  there is no unique or exclusion constraint matching the
-- ->         ON CONFLICT specification
```

A sync job should also report what it did, and should not rewrite rows that did not change: every rewrite costs disk work, changes updated_at, and can wake up downstream jobs that watch for changes. Two additions do this. A WHERE on the DO UPDATE skips the update when nothing differs. And RETURNING (xmax = 0) tells an insert from an update.

In [ ]:
# Concept 2: an upsert that says what it did: inserted, updated, unchanged.
import os

import psycopg

DB_URL = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")

UPSERT = """
INSERT INTO lesson_entries AS e (uid, title, status)
VALUES (%s, %s, %s)
ON CONFLICT (uid) DO UPDATE
    SET title = EXCLUDED.title, status = EXCLUDED.status,
        updated_at = now()
    -- only touch the row if something really changed; IS DISTINCT FROM
    -- is "!=" that also treats NULL = NULL as equal (Day 16's NULL rules)
    WHERE (e.title, e.status) IS DISTINCT FROM
          (EXCLUDED.title, EXCLUDED.status)
-- xmax is a hidden system column: 0 means this statement created the row.
-- A widely used trick; Postgres 18 can say it directly with RETURNING old.
RETURNING (xmax = 0) AS inserted
"""


def upsert(conn, uid: str, title: str, status: str) -> str:
    row = conn.execute(UPSERT, (uid, title, status)).fetchone()
    if row is None:  # the WHERE skipped the update, so nothing came back
        return "unchanged"
    return "inserted" if row[0] else "updated"


with psycopg.connect(DB_URL, autocommit=True) as conn:
    print(upsert(conn, "blt110", "Store finder", "draft"))      # -> inserted
    print(upsert(conn, "blt110", "Store finder", "draft"))      # -> unchanged
    print(upsert(conn, "blt110", "Store finder", "published"))  # -> updated
    # Run it 100 times: same final state. That is what "idempotent" means.
    for _ in range(100):
        upsert(conn, "blt110", "Store finder", "published")
    print(conn.execute(
        "SELECT count(*), max(status) FROM lesson_entries "
        "WHERE uid = 'blt110'").fetchone())  # -> (1, 'published')

IS DISTINCT FROM matters here. With plain <>, comparing NULL to NULL gives unknown, so a row whose word count is NULL on both sides would look "changed" or "unchanged" depending on how you wrote it. IS DISTINCT FROM treats two NULLs as equal and NULL versus a value as different, which is what a sync needs.

FDE TIP: in Postgres 15 and later there is also MERGE, the SQL-standard version that can insert, update and delete in one statement. ON CONFLICT is still the everyday tool for upserts, and it is what you will see in most customer code.

### Real-world FDE example

A RAG pilot for a support team re-indexed 40,000 help-centre articles every night by deleting the vector table and inserting everything again. Search went empty for 20 minutes each night, and the embeddings bill was paid in full every time. The rewrite upserted on (article_id, chunk_no) and only re-embedded articles whose content hash changed: about 300 a night. Same result, no gap, and a fraction of the cost. You will build this in Week 6.

### Common mistakes

- Writing ON CONFLICT on a column with no UNIQUE constraint. Postgres refuses: the target must be a unique constraint or primary key.

- Expecting DO NOTHING ... RETURNING to give you the existing row's id. It returns nothing for a skipped row; use DO UPDATE, or a SELECT afterwards.

- Sending the same key twice in one multi-row INSERT ... ON CONFLICT DO UPDATE. Postgres refuses with "cannot affect row a second time" (Check your understanding, question 4). De-duplicate the batch first.

- SET-ting every column including the ones the export does not contain, and wiping values another system owns.

- Upserting without a change check, so every nightly run rewrites every row and updated_at stops meaning anything.

## Concept 3: Indexes, how a database finds rows fast

### Why it matters for an FDE

"The dashboard is slow" is one of the most common tickets you will get on a customer site, and a missing index is the most common cause. The query was fine with 1,000 rows in the demo and takes 8 seconds with 5 million in production. Adding the right index is often a one-line fix that takes a page from seconds to milliseconds. Adding the wrong ones slows down every write and wastes disk. You need to know which is which.

### How it works

Without an index, Postgres finds matching rows with a sequential scan: it reads every row of the table and keeps the ones that match, like reading a whole book to find one word. An index is a separate, sorted structure that points at rows, like the index at the back of the book. The default kind is a B-tree: keys kept in sorted order in a shallow tree, so finding one key among millions takes a handful of steps of binary search (today's DSA problem is exactly that step).

You already have some: every primary key and UNIQUE constraint created one. For other columns you write CREATE INDEX name ON table (columns). Postgres then keeps the index up to date on every INSERT, UPDATE and DELETE, which is the price you pay.

In [ ]:
# Concept 3: the same query before and after an index, timed from Python.
import os
import time

import psycopg

DB_URL = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")

QUERY = """
SELECT sum(views) FROM lesson_views
WHERE entry_uid = %s AND viewed_on >= %s
"""
ARGS = ("blt0042", "2026-06-01")


def timed(conn) -> tuple[int, float]:
    start = time.perf_counter()  # like performance.now() in JS
    total = conn.execute(QUERY, ARGS).fetchone()[0]
    for _ in range(19):  # run it 20 times so one slow run does not mislead
        conn.execute(QUERY, ARGS).fetchone()
    ms = (time.perf_counter() - start) * 1000 / 20
    return total, round(ms, 2)


with psycopg.connect(DB_URL, autocommit=True) as conn:
    conn.execute("DROP INDEX IF EXISTS lesson_views_uid_day_idx")
    print("no index:  ", timed(conn))  # -> no index:   (25536, 10.77)
    # A B-tree index on the two columns the WHERE filters on, in that order.
    conn.execute(
        "CREATE INDEX lesson_views_uid_day_idx "
        "ON lesson_views (entry_uid, viewed_on)"
    )
    print("with index:", timed(conn))  # -> with index: (25536, 0.2)
# Same answer, about 50 times faster. Your milliseconds will differ;
# the size of the gap is the point.

An index on several columns, a composite index, is sorted by the first column, then by the second within each first value. That order decides which queries it can serve. It helps a query that filters on the first column, or on the first and second together. It does not help a query that filters only on the second column, in the same way a phone book sorted by surname cannot find every "Rohit" quickly.

```sql
-- Concept 3: which queries can use the index on (entry_uid, viewed_on)?
-- Run after the previous example, so the index exists. EXPLAIN shows the
-- plan without running the query; Concept 4 reads plans properly.
-- Your cost numbers may differ slightly (statistics are sampled).

-- 1. Filter on the FIRST column of the index: the index is used.
EXPLAIN SELECT * FROM lesson_views WHERE entry_uid = 'blt0042';
-- -> Bitmap Heap Scan on lesson_views  (cost=17.06..1287.46 rows=598 ...)
-- ->   ->  Bitmap Index Scan on lesson_views_uid_day_idx

-- 2. Filter on the SECOND column only: not used. A phone book sorted by
--    surname, then first name, cannot find "every Rohit" quickly.
EXPLAIN SELECT * FROM lesson_views WHERE viewed_on = '2026-06-01';
-- -> Gather  (cost=1000.00..5166.58 rows=497 ...)
-- ->   ->  Parallel Seq Scan on lesson_views

-- 3. A function around the column hides it from the index.
EXPLAIN SELECT * FROM lesson_views WHERE lower(entry_uid) = 'blt0042';
-- -> Gather  (cost=1000.00..5708.06 rows=1500 ...)
-- ->   ->  Parallel Seq Scan on lesson_views
-- FDE TIP: if you really need lower(), index the expression itself:
-- CREATE INDEX ... ON lesson_views (lower(entry_uid));
```

Rules of thumb for choosing indexes: index the columns your important queries filter or join on; in a composite index put the column you compare with = first and the range column (>=, BETWEEN) after it; skip indexes on tiny tables and on columns with only a few distinct values; and remember that each index costs space and write time. The size check in Concept 4 shows today's index is more than half the size of the table.

### Real-world FDE example

A media customer's editor dashboard listed "my drafts, newest first" and took 6 seconds for their 3 million entries. The query filtered on author_id and status and sorted by updated_at. One index on (author_id, status, updated_at) made it 4 milliseconds, because Postgres could jump to that author's drafts and read them already in order. The same team had also added 11 single-column indexes "just in case"; removing the 7 that no query used made their nightly import 30% faster.

### Common mistakes

- Indexing every column. Each index slows every write and uses disk; add them for real queries you have measured.

- Wrong column order in a composite index, so the query that needs it cannot use it.

- Wrapping the column in a function (lower(email), date(created_at)) and wondering why the index is ignored. Index the expression, or compare the raw column.

- Forgetting that foreign key columns are not indexed automatically in Postgres. Joins and cascading deletes on author_id need an index you create yourself.

- Creating an index on a large, busy production table without CONCURRENTLY and blocking writes for minutes (Concept 5).

## Concept 4: Reading query plans with EXPLAIN and EXPLAIN ANALYZE

### Why it matters for an FDE

Guessing is how people end up with 11 unused indexes. Postgres will tell you exactly how it runs a query and where the time goes. Reading a plan is the skill that lets you walk into a "the system is slow" escalation, find the cause in ten minutes, and show the customer before-and-after numbers they can trust. Those numbers also go straight into your portfolio write-ups.

### How it works

Every query goes through the planner, which considers different ways to run it and picks the one it estimates is cheapest, using statistics about each table (how many rows, how many distinct values). EXPLAIN query shows the chosen plan without running it. EXPLAIN ANALYZE query runs it and adds what really happened.

A plan is a tree. Read it from the most indented line outwards: the innermost node runs first and feeds its rows to the line above. On each line, cost is the planner's estimate in its own units (useful only for comparing plans), rows is how many rows it expected, and in ANALYZE output actual time is in milliseconds (time to first row .. time to last row) and actual rows is what really came out. At the bottom, Execution Time is the number you report.

```sql
-- Concept 4: EXPLAIN ANALYZE runs the query and reports what happened.
-- Re-run setup.sql first (no index). Plan lines are shortened here, and
-- your costs and times will differ a little: compare the shapes.
SET max_parallel_workers_per_gather = 0;  -- one process: simpler plans

EXPLAIN ANALYZE
SELECT sum(views) FROM lesson_views
WHERE entry_uid = 'blt0042' AND viewed_on >= '2026-06-01';
-- -> Aggregate  (cost=6411.21..6411.22 rows=1 width=8)
-- ->            (actual time=11.655..11.656 rows=1 loops=1)
-- ->   ->  Seq Scan on lesson_views  (cost=0.00..6411.00 rows=84 width=4)
-- ->                    (actual time=9.152..11.642 rows=84 loops=1)
-- ->         Filter: ((viewed_on >= '2026-06-01'::date)
-- ->                  AND (entry_uid = 'blt0042'::text))
-- ->         Rows Removed by Filter: 299916
-- -> Planning Time: 0.250 ms
-- -> Execution Time: 11.703 ms

CREATE INDEX lesson_views_uid_day_idx
    ON lesson_views (entry_uid, viewed_on);

EXPLAIN ANALYZE
SELECT sum(views) FROM lesson_views
WHERE entry_uid = 'blt0042' AND viewed_on >= '2026-06-01';
-- -> Aggregate  (cost=286.86..286.87 rows=1 width=8)
-- ->            (actual time=0.136..0.137 rows=1 loops=1)
-- ->   ->  Bitmap Heap Scan on lesson_views  (... rows=84 ...)
-- ->         Recheck Cond: ((entry_uid = 'blt0042'::text)
-- ->                        AND (viewed_on >= '2026-06-01'::date))
-- ->         Heap Blocks: exact=84
-- ->         ->  Bitmap Index Scan on lesson_views_uid_day_idx
-- ->               (actual time=0.040..0.040 rows=84 loops=1)
-- -> Planning Time: 0.192 ms
-- -> Execution Time: 0.167 ms

-- What did the index cost? Disk space (and work on every write).
SELECT pg_size_pretty(pg_relation_size('lesson_views')) AS table_size,
       pg_size_pretty(pg_relation_size('lesson_views_uid_day_idx'))
           AS index_size;
-- ->  table_size | index_size
-- ->  15 MB      | 9264 kB
```

How to read the first plan: the innermost node is Seq Scan on lesson_views, and Rows Removed by Filter: 299916 says it read 300,000 rows to keep 84. That ratio is the signature of a missing index. In the second plan the Bitmap Index Scan finds the 84 row locations in the index first, then the Bitmap Heap Scan fetches only those rows. The same 84 rows come out, about 70 times faster.

Three more things to look for. When estimated rows and actual rows are far apart (expected 5, got 50,000), the statistics are stale and the planner chose badly: run ANALYZE table and look again. An Index Scan reads rows in index order; a Bitmap scan collects matches first and then reads the table pages in order, which is better when there are many matches; both mean the index is used. And for a tiny table a Seq Scan is the right choice, so do not fight it.

```sql
-- Concept 4: EXPLAIN ANALYZE really runs the statement. For a DELETE or
-- UPDATE, wrap it in a transaction and roll back.
BEGIN;
EXPLAIN ANALYZE DELETE FROM lesson_views WHERE entry_uid = 'blt0042';
-- -> Delete on lesson_views  (actual time=1.181..1.182 rows=0 loops=1)
-- ->   ->  Bitmap Heap Scan on lesson_views  (... rows=600 loops=1)
-- -> Execution Time: 1.245 ms
ROLLBACK;  -- the 600 rows come back
SELECT count(*) FROM lesson_views WHERE entry_uid = 'blt0042';
-- -> 600
-- GOTCHA: without BEGIN ... ROLLBACK those 600 rows would be gone.
```

### Real-world FDE example

A customer said their search API "randomly" took 9 seconds. EXPLAIN ANALYZE on the slow case showed the planner expected 12 rows and got 410,000, then chose a nested loop that ran 410,000 times. The table had grown 40 times since the last ANALYZE after a bulk import. Running ANALYZE fixed the plan in seconds, and the import job got an ANALYZE step at its end, the same line you see at the end of today's seed_views.sql.

### Common mistakes

- Running EXPLAIN ANALYZE on an UPDATE or DELETE outside a transaction and changing real data.

- Treating cost as milliseconds. Cost is the planner's estimate; actual time is the measurement.

- Measuring once. The first run reads from disk and later runs hit the cache; run a query a few times and compare like with like.

- Testing on a table with 50 rows and concluding "the index is not used". The planner is right to scan a tiny table.

- Forgetting ANALYZE after a big load, so the planner works from statistics about a table that no longer exists.

## Concept 5: Migrations, schema changes as versioned files

### Why it matters for an FDE

On a customer engagement the same schema lives in at least three places: your laptop, their staging database and their production database, plus every teammate's machine. If schema changes are typed by hand into psql, those copies drift apart within a week, and "works on staging, fails in prod" follows. Migrations make the schema reproducible: the database's structure is code, reviewed in pull requests like any other code.

### How it works

A migration is one file describing one change, numbered so the order is fixed. A runner keeps a bookkeeping table in the database listing which files have run. Each time it starts, it runs only the files not yet in that list, in order, and records each one. A fresh database gets every migration from 001; production, already at 002, gets only 003 and 004. It is the same idea as Git commits for your schema, or Contentstack's content-type changes being promoted from one environment to the next instead of being clicked again by hand.

```bash
# Concept 5: the migrations folder for today's task. One numbered file
# per change; the number decides the order; the name says what it does.
day19/migrations/
    001_create_cms_tables.sql   # given: authors, posts, tags, links
    002_create_cms_views.sql    # given: daily views, no index yet
    003_add_views_index.sql     # YOU write it, after measuring (Step 4)
    004_title_not_blank.sql     # YOU write it: a new CHECK rule

# The runner keeps a list of what already ran, in the database itself:
#   schema_migrations
#    version                    | applied_at
#    001_create_cms_tables.sql  | 2026-11-12 09:14:03+05:30
#    002_create_cms_views.sql   | 2026-11-12 09:14:03+05:30
# Next run: 001 and 002 are in the list, so only 003 and 004 run.
# REMEMBER: never edit a migration that has run anywhere. Add a new one.
```

Two Postgres facts make runners safer. Schema changes (DDL) are transactional, so a migration and its bookkeeping row can succeed or fail together, which is what you will build in apply_migrations. And some operations are special: CREATE INDEX CONCURRENTLY cannot run in a transaction at all.

```sql
-- Concept 5: two facts about schema changes in Postgres.

-- 1. DDL (CREATE, ALTER, DROP) is transactional: a failed migration
--    can be rolled back completely. MySQL cannot do this.
BEGIN;
ALTER TABLE lesson_entries ADD COLUMN locale text NOT NULL DEFAULT 'en-us';
SELECT uid, locale FROM lesson_entries ORDER BY uid LIMIT 2;
-- ->   uid   | locale
-- ->  blt101 | en-us
-- ->  blt102 | en-us
ROLLBACK;
SELECT count(*) FROM information_schema.columns
WHERE table_name = 'lesson_entries' AND column_name = 'locale';
-- -> 0      (the column is gone again: the ALTER was rolled back)

-- 2. On a big, busy table, a normal CREATE INDEX blocks writes until it
--    finishes. CONCURRENTLY does not block, but cannot run in a
--    transaction, so a migration runner must run it on its own.
BEGIN;
CREATE INDEX CONCURRENTLY lesson_views_country_idx
    ON lesson_views (country);
-- -> ERROR:  CREATE INDEX CONCURRENTLY cannot run inside a transaction
-- ->         block
ROLLBACK;
CREATE INDEX CONCURRENTLY IF NOT EXISTS lesson_views_country_idx
    ON lesson_views (country);
-- -> CREATE INDEX
-- IF NOT EXISTS makes the file safe to run twice; run it again:
-- -> NOTICE:  relation "lesson_views_country_idx" already exists, skipping
```

In real projects you will use a migration tool rather than writing your own: Alembic with SQLAlchemy in Python (you will meet it in Week 4), Flyway or Liquibase in Java shops, Prisma or Knex in Node teams. They all work like today's runner: numbered files plus a bookkeeping table. Writing a small one yourself once is the fastest way to understand what those tools do and how they fail.

### Real-world FDE example

An FDE team shipped a fix to a customer's staging database by running an ALTER TABLE in psql and forgot to write it down. Two weeks later the production deploy failed on a column that "definitely exists". After that, the team rule was: no schema change outside a numbered migration, every migration reviewed in a pull request, and the runner applied in CI against a fresh database on every push, so a broken migration fails a build instead of a release.

### Common mistakes

- Editing a migration that has already run somewhere. The runner sees the name in its table and skips it, so the change never reaches that database. Write a new file instead.

- Two teammates both creating 003_... on different branches. Agree the next number when you open the PR, or use timestamps as numbers.

- Recording a migration as applied even though it failed halfway. Run the file and its bookkeeping row in one transaction.

- Putting CREATE INDEX CONCURRENTLY in a migration that the runner wraps in a transaction.

- Seeding test data inside migrations. Migrations change structure; seed data belongs in separate scripts like seed_views.sql.

## Check your understanding (predict first, then run)

Write down what each numbered part prints, then run the file inside day19/ with freshly created lesson tables. Answers are on the last page.

In [ ]:
# Predict, then run. Re-run setup.sql before you start.
import psycopg

conn = psycopg.connect("postgresql://localhost/fde", autocommit=True)
ADD = "INSERT INTO lesson_entries (uid, title, slug, word_count) VALUES "

# 1: does this insert succeed? The CHECK is word_count >= 0.
conn.execute(ADD + "('blt301', 'A', 'a', NULL)")
print(conn.execute("SELECT count(*) FROM lesson_entries "
                   "WHERE uid = 'blt301'").fetchone())

# 2: slug is UNIQUE. Do both of these succeed?
conn.execute(ADD + "('blt302', 'B', NULL, 1)")
conn.execute(ADD + "('blt303', 'C', NULL, 1)")
print(conn.execute("SELECT count(*) FROM lesson_entries "
                   "WHERE slug IS NULL").fetchone())

# 3: what does fetchone() return here?
cur = conn.execute(ADD + "('blt101', 'X', 'x', 1) "
                   "ON CONFLICT (uid) DO NOTHING RETURNING uid")
print(cur.fetchone())

# 4: the same uid twice in ONE statement. What happens?
try:
    conn.execute(ADD + "('blt304', 'D', 'd', 1), ('blt304', 'E', 'e', 2) "
                 "ON CONFLICT (uid) DO UPDATE SET title = EXCLUDED.title")
except psycopg.Error as e:
    print(type(e).__name__, e)

## Hands-on task (about 2 hours): day19/upsert_loader.py

Acme Retail runs your Day 18 loader every night after the CMS export. On night two it failed: every post was already there. The customer's requirements for the replacement: it can run any number of times; it reports how many posts were inserted, updated and unchanged; a bad entry stops the whole run with a message their content team understands, and leaves the database untouched; and the "daily views" report their editors use must get fast.

```sql
# day19/ layout for today's task. Everything except upsert_loader.py's
# function bodies, migrations 003 and 004, explain_notes.md and dsa.py
# is given.
day19/
    setup.sql                 # lesson tables for the concept examples
    reset.sql                 # drops the task tables: a clean start
    migrations/
        001_create_cms_tables.sql
        002_create_cms_views.sql
    seed_views.sql            # 400,000 rows of daily views
    slow_query.sql            # the report the team calls "slow"
    export_v1.json            # 6 posts, the nightly CMS export
    export_v2.json            # next night: 1 new post, 2 edited
    export_bad.json           # one post with status "live"
    upsert_loader.py          # YOUR loader: 6 TODO functions
    test_upsert_loader.py     # 21 given tests
    explain_notes.md          # YOUR before/after notes
    requirements.txt          # psycopg[binary] 3.3 and pytest
    dsa.py                    # today's two DSA problems
```

Step 1 (10 min): read migrations 001 and 002 and list every rule they define. For each, write one export value that would break it.

Step 2 (25 min), no database yet: write read_export. Then apply_migrations, and run python -m pytest -q -k "read_export or migration".

Step 3 (50 min): write upsert_author, upsert_post, sync_tags, load and main, then do the first four manual runs below.

Step 4 (10 min): seed the views table and run slow_query.sql. Paste the output into explain_notes.md and fill in the three lines under it.

Step 5 (15 min): write migrations/003_add_views_index.sql (an index named cms_views_post_day_idx that serves slow_query.sql) and migrations/004_title_not_blank.sql (a CHECK named cms_posts_title_not_blank that refuses titles that are empty or only spaces). Run the loader again so it applies them.

Step 6 (10 min): run slow_query.sql again, finish explain_notes.md, then run the whole test suite.

```bash
# Steps 2 to 6: what correct runs look like, in order.
$ psql "$DATABASE_URL" -f reset.sql
$ python upsert_loader.py export_v1.json
migrations applied: 001_create_cms_tables.sql, 002_create_cms_views.sql
export_v1.json: inserted 6, updated 0, unchanged 0
$ python upsert_loader.py export_v1.json     # the re-run that broke Day 18
migrations: up to date
export_v1.json: inserted 0, updated 0, unchanged 6
$ python upsert_loader.py export_v2.json
migrations: up to date
export_v2.json: inserted 1, updated 2, unchanged 4
$ python upsert_loader.py export_bad.json; echo "exit $?"
migrations: up to date
error: entry blt008 breaks rule cms_posts_status_check; nothing was loaded
exit 1
$ psql "$DATABASE_URL" -f seed_views.sql
$ psql "$DATABASE_URL" -f slow_query.sql     # BEFORE: paste into notes
#   (write migrations 003 and 004 now)
$ python upsert_loader.py export_v2.json
migrations applied: 003_add_views_index.sql, 004_title_not_blank.sql
export_v2.json: inserted 0, updated 0, unchanged 7
$ psql "$DATABASE_URL" -f slow_query.sql     # AFTER: paste into notes
```

Acceptance criteria:

- All the runs above print exactly those lines and exit with those codes (echo $?). Errors go to stderr, and no run prints a traceback.

- After the export_bad.json run, the tables hold exactly what export_v2.json left: posts 7, authors 4, tags 8, tag links 11.

- python -m pytest -q shows 21 passed (17 pass before migrations 003 and 004 exist).

- Every value reaches SQL through a placeholder; no f-string or + builds SQL text. Each migration runs in its own transaction together with its bookkeeping row; the whole load runs in one conn.transaction() block.

- explain_notes.md holds both plans: a Seq Scan before 003 and a Bitmap Index Scan or Index Scan after it, the two execution times, and your two sentences on the column order.

- Migrations 001 and 002 are unchanged; 003 and 004 each make one change, and after the run all four names are in schema_migrations.

- python -c "import upsert_loader" inside day19/ prints nothing.

- Committed and pushed with a meaningful message; .venv/ is not committed.

Hints (no solutions, try first):

- upsert_post is Concept 2's second example with six columns. Named placeholders (%(title)s with a dict) keep it readable.

- For "unchanged", the change check must compare every column you write, including author_id and published_at, and must not compare updated_at.

- sync_tags is three small steps: clean the names, upsert each tag and its link, delete the links you did not keep. The test file shows exactly what "clean" means.

- If apply_migrations records a file as applied even when it fails, the INSERT into schema_migrations is outside the transaction block.

- If tests hang or fail with "current transaction is aborted", an exception was caught inside a transaction block and the code carried on (Day 18, Concept 3).

- For migration 003, look at the WHERE of slow_query.sql: which column is compared with =, and which with >=? Concept 3's rules of thumb decide the order.

- For migration 004, btrim(title) removes spaces from both ends, like .trim() in JavaScript.

## DSA practice (30 min)

Both problems are in day19/dsa.py with signatures, examples and hints.

- In-memory upsert (easy). upsert_records(store, incoming) merges records into a dict keyed by uid and returns the inserted, updated and unchanged counts: the same contract as today's loader, without a database. Hint: three cases, decided by in and ==.

- Lower bound (medium). lower_bound(keys, target) returns the first position whose key is >= target in a sorted list, in O(log n). This is the search a B-tree index does at every level. Hint: keep lo and hi, and move hi = mid (not mid - 1) when keys[mid] is not smaller than the target. Check yourself with bisect.bisect_left afterwards.

## Quiz (3 quick questions)

- The customer's admin UI already validates that status is draft, published or archived. Give two reasons to add the CHECK constraint to the database anyway.

- A report filters WHERE country = 'IN' AND viewed_on >= '2026-01-01'. Your teammate proposes an index on (viewed_on, country). What would you suggest, and how would you prove it is better?

- Your nightly job runs twice by mistake. With today's loader, what is in the database after the second run, and what do the two summary lines say?

## What you achieved today

Your database now enforces its own rules, your loader survives re-runs and retries, you can prove a performance fix with a query plan instead of a guess, and your schema changes are versioned files that any database can replay. Idempotent syncs, data rules and "why is it slow" are three things every FDE handles in their first month on a customer site.

Tomorrow (Day 20): analytics SQL: CTEs, window functions for "latest version per entry" and "rank by views", JSONB columns for searching inside CMS fields, and a pandas summary built from SQL.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 19 review": your recap answers, your four predictions next to the real outputs, upsert_loader.py, migrations 003 and 004, explain_notes.md, the pytest summary line, the terminal output of the manual runs, your quiz answers, dsa.py, and the link to your latest commit. Say how far the Day 18 catch-up got.

## Recap answers

*Your answers:*
